# Verifikasi perbaikan adapter dan SENA v0.5

Notebook ini benar-benar menjalankan pemeriksaan kontrak dan inferensi lokal CPU. Target dan cutoff 0,95 tidak diubah. Replay field menggunakan prediksi mentah dan acuan manusia lama yang tetap disimpan. Inferensi berikutnya memakai bobot SENA baru yang hash-nya diperiksa dan bobot NARA kandidat yang sama.

Lima puluh contoh lama telah dipakai untuk memperbaiki adapter sehingga merupakan data regresi pengembangan. Dua puluh empat narasi tambahan memiliki acuan buatan asisten dan hanya dipakai sekali setelah pemilihan checkpoint. Kedua hasil tidak memberi persetujuan mutu manusia secara otomatis. Encoder tidak dilatih ulang di notebook ini. Satu bias head NON_EVENT dipilih dari data pengembangan sebelum pengujian; ini fitting parameter pengembangan dan bukan bukti kalibrasi probabilitas atau validasi independen.

In [ ]:
from pathlib import Path
import os,sys,subprocess,json
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT'])
ASSETS=Path(os.environ['SLR09_ASSET_DIR'])
BASE=Path(os.environ['SLR09_TRAIN_OUTPUT']).parent
PREVIOUS=ROOT/'evidence/quality_fix_v05/previous_50'
SENA_DIR=Path(os.environ.get('SLR09_QUALITY_SENA_DIR',str(BASE/'prior_adjustment_20261019_cpu/best_model')))
OUTPUT=BASE/'quality_diagnostics_v05'
def execute(command):
 p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
 for line in p.stdout: print(line,end='',flush=True)
 p.wait()
 if p.returncode: raise RuntimeError('Pemeriksaan gagal: '+str(p.returncode))
print('Inference device: CPU. Target dan acuan lama tetap.')

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/fit_sena_prior.py'),
 '--source',str(BASE/'training_run_20261019_cpu_focus'),
 '--output',str(BASE/'prior_adjustment_20261019_cpu')])
execute([sys.executable,str(ROOT/'code/quality_fix/prepare_quality_candidate.py'),
 '--run',str(BASE/'prior_adjustment_20261019_cpu'),'--exports',str(BASE/'exports')])

In [ ]:
execute([sys.executable,str(ROOT/'code/run_active_tests.py'),'--evidence-dir',str(BASE/'contract_tests_v05')])

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/replay_fields.py'),
 '--source',str(PREVIOUS),'--output',str(BASE/'field_replay_final_v05')])

In [ ]:
execute([sys.executable,str(ROOT/'code/quality_fix/run_quality_diagnostics.py'),
 '--previous',str(PREVIOUS),'--challenge',str(ROOT/'data/quality_fix_fresh_challenge_v0.5.json'),
 '--sena-dir',str(SENA_DIR),'--nara-archive',str(ASSETS/'SILAR_NARA_reviewed_pilot_20261018T001122_607337Z.zip'),
 '--output',str(OUTPUT)])

In [ ]:
report=json.loads((OUTPUT/'evaluation.json').read_text())
state=json.loads((OUTPUT/'run_state.json').read_text())
assert state['status']=='SUCCEEDED' and state['model_inference_executed']
assert not state['quality_acceptance_approved'] and not state['card_done']
print('Hasil nyata tersimpan. Status mutu memerlukan penilaian terpisah.')
print(json.dumps({'checks':report['checks'],'quality_acceptance_approved':False,'card_done':False},indent=2))